<h1>Create K-Fold datasets</h1>

In [1]:
# Make sure you have tqdm installed. If not, you can run this line in a separate cell:
# !pip install tqdm

import os
import shutil
from tqdm.notebook import tqdm # Use the notebook-friendly version of tqdm

# ==============================================================================
# --- 1. Configuration: Adjust these parameters as needed ---
# ==============================================================================

# Define file paths
input_file = './fds-challenge-dataset/train.jsonl'
output_dir = './fds-challenge-dataset/kfolds'
temp_dir = './fds-challenge-dataset/temp_chunks' # Directory for temporary files

# Number of folds for cross-validation
n_splits = 10


# ==============================================================================
# --- 2. Helper Functions ---
# ==============================================================================

def get_line_count(filename):
    """Counts lines in a file efficiently without loading it into memory."""
    print("First, counting the total number of lines for the progress bar...")
    try:
        with open(filename, 'rb') as f:
            # A fast way to count lines
            count = sum(1 for _ in f)
        print(f"Found {count:,} lines in the input file.")
        return count
    except FileNotFoundError:
        return -1 # Return an error code

def create_kfold_splits():
    """
    Creates k-fold splits for a very large file without loading it into memory.
    """
    # 1. Setup: Create directories
    print("--- Step 1: Setting up directories ---")
    os.makedirs(output_dir, exist_ok=True)
    # Clean up old temp files if they exist from a previous failed run
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir, exist_ok=True)
    print(f"Output will be in: {output_dir}")
    print(f"Temporary files will be in: {temp_dir}")

    # Get total lines for tqdm progress bar
    total_lines = get_line_count(input_file)
    if total_lines == -1:
        print(f"Error: Input file not found at '{input_file}'")
        return

    # 2. First Pass: Distribute lines into temporary chunk files
    print(f"\n--- Step 2: Distributing {total_lines:,} lines into {n_splits} temporary chunks ---")
    
    try:
        # Open all temporary file writers at once
        chunk_files = [open(os.path.join(temp_dir, f'chunk_{i}.jsonl'), 'w', encoding='utf-8') for i in range(n_splits)]
        
        with open(input_file, 'r', encoding='utf-8') as f_in:
            # Use tqdm for a progress bar
            for i, line in enumerate(tqdm(f_in, total=total_lines, desc="Splitting to chunks")):
                # Assign line to a chunk file in a round-robin fashion
                target_chunk_index = i % n_splits
                chunk_files[target_chunk_index].write(line)
    finally:
        # Ensure all files are closed, even if an error occurs
        for f in chunk_files:
            f.close()
    
    print("Finished distributing lines to temporary chunks.")

    # 3. Second Pass: Assemble final train/validation files from chunks
    print(f"\n--- Step 3: Assembling final {n_splits} train/validation folds ---")
    for k in tqdm(range(n_splits), desc="Assembling Folds"):
        fold_num = k + 1
        
        # Define final output file paths for this fold
        train_output_file = os.path.join(output_dir, f'fold_{fold_num}_train.jsonl')
        val_output_file = os.path.join(output_dir, f'fold_{fold_num}_val.jsonl')

        # The validation file is just the corresponding chunk. We copy it.
        val_chunk_path = os.path.join(temp_dir, f'chunk_{k}.jsonl')
        shutil.copy(val_chunk_path, val_output_file)

        # The training file is the concatenation of all *other* chunks
        with open(train_output_file, 'wb') as f_train_out: # Open in binary write mode for efficiency
            for i in range(n_splits):
                if i == k:
                    continue # Skip the validation chunk
                
                chunk_to_append_path = os.path.join(temp_dir, f'chunk_{i}.jsonl')
                with open(chunk_to_append_path, 'rb') as f_chunk_in: # Open in binary read mode
                    shutil.copyfileobj(f_chunk_in, f_train_out)
    
    print("Finished assembling all folds.")

    # 4. Cleanup: Remove the temporary directory and its contents
    print("\n--- Step 4: Cleaning up temporary files ---")
    shutil.rmtree(temp_dir)
    print(f"Removed temporary directory: {temp_dir}")

    print("\n✅ Process finished successfully!")


# ==============================================================================
# --- 3. Main Execution Logic ---
# ==============================================================================

# Check if the input file exists before starting the whole process
if not os.path.exists(input_file):
    print(f"❌ Error: Input file not found at '{input_file}'")
    print("Please make sure the path is correct and the file is in place.")
else:
    create_kfold_splits()

--- Step 1: Setting up directories ---
Output will be in: ./fds-challenge-dataset/kfolds
Temporary files will be in: ./fds-challenge-dataset/temp_chunks
First, counting the total number of lines for the progress bar...
Found 100,000 lines in the input file.

--- Step 2: Distributing 100,000 lines into 10 temporary chunks ---


Splitting to chunks:   0%|          | 0/100000 [00:00<?, ?it/s]

KeyboardInterrupt: 

Pokemon features structure

In [1]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126

import torch
import torch.nn as nn

class PokemonBattleModel(nn.Module):
    def __init__(self):
        super(PokemonBattleModel, self).__init__()

        # Define the structure of the input vector with descriptive names for P1's Pokemon
        # Features for P1's first Pokemon
        self.p1_pokemon1_name_features = 151  # One-hot encoding for 151 possible Pokemon names
        self.p1_pokemon1_level_feature = 1   # Numerical feature for the level
        self.p1_pokemon1_types_features = 40 # One-hot encoding for types (assuming max 2 types * 20 possible types)
        self.p1_pokemon1_base_stats = 6      # Numerical features for base stats (HP, Attack, Defense, Special Attack, Special Defense, Speed)

        # Features for P1's second Pokemon
        self.p1_pokemon2_name_features = 151
        self.p1_pokemon2_level_feature = 1
        self.p1_pokemon2_types_features = 40
        self.p1_pokemon2_base_stats = 6

        # Features for P1's third Pokemon
        self.p1_pokemon3_name_features = 151
        self.p1_pokemon3_level_feature = 1
        self.p1_pokemon3_types_features = 40
        self.p1_pokemon3_base_stats = 6

        # Features for P1's fourth Pokemon
        self.p1_pokemon4_name_features = 151
        self.p1_pokemon4_level_feature = 1
        self.p1_pokemon4_types_features = 40
        self.p1_pokemon4_base_stats = 6

        # Features for P1's fifth Pokemon
        self.p1_pokemon5_name_features = 151
        self.p1_pokemon5_level_feature = 1
        self.p1_pokemon5_types_features = 40
        self.p1_pokemon5_base_stats = 6

        # Features for P1's sixth Pokemon
        self.p1_pokemon6_name_features = 151
        self.p1_pokemon6_level_feature = 1
        self.p1_pokemon6_types_features = 40
        self.p1_pokemon6_base_stats = 6

        # Features for P2's first Pokemon
        self.p2_pokemon1_name_features = 151
        self.p2_pokemon1_level_feature = 1
        self.p2_pokemon1_types_features = 40
        self.p2_pokemon1_base_stats = 6

        # Features for P1's current Pokemon in Turn 1
        self.turn1_p1_pokemon_name = 151
        self.turn1_p1_pokemon_hp_pct = 1
        self.turn1_p1_pokemon_status = 8
        self.turn1_p1_pokemon_effects = 2
        self.turn1_p1_pokemon_boosts = 6
        self.turn1_p1_pokemon_move_name = 165
        self.turn1_p1_pokemon_move_type = 20
        self.turn1_p1_pokemon_move_category = 3
        self.turn1_p1_pokemon_move_stats = 3 # Power, Accuracy, Priority

        # Features for P2's current Pokemon in Turn 1
        self.turn1_p2_pokemon_name = 151
        self.turn1_p2_pokemon_hp_pct = 1
        self.turn1_p2_pokemon_status = 8
        self.turn1_p2_pokemon_effects = 2
        self.turn1_p2_pokemon_boosts = 6
        self.turn1_p2_pokemon_move_name = 165
        self.turn1_p2_pokemon_move_type = 20
        self.turn1_p2_pokemon_move_category = 3
        self.turn1_p2_pokemon_move_stats = 3 # Power, Accuracy, Priority

        # Define other feature group sizes here as needed

        # Calculate the total number of input features
        self.total_input_features = (self.p1_pokemon1_name_features + self.p1_pokemon1_level_feature + self.p1_pokemon1_types_features + self.p1_pokemon1_base_stats +
                                     self.p1_pokemon2_name_features + self.p1_pokemon2_level_feature + self.p1_pokemon2_types_features + self.p1_pokemon2_base_stats +
                                     self.p1_pokemon3_name_features + self.p1_pokemon3_level_feature + self.p1_pokemon3_types_features + self.p1_pokemon3_base_stats +
                                     self.p1_pokemon4_name_features + self.p1_pokemon4_level_feature + self.p1_pokemon4_types_features + self.p1_pokemon4_base_stats +
                                     self.p1_pokemon5_name_features + self.p1_pokemon5_level_feature + self.p1_pokemon5_types_features + self.p1_pokemon5_base_stats +
                                     self.p1_pokemon6_name_features + self.p1_pokemon6_level_feature + self.p1_pokemon6_types_features + self.p1_pokemon6_base_stats +
                                     self.p2_pokemon1_name_features + self.p2_pokemon1_level_feature + self.p2_pokemon1_types_features + self.p2_pokemon1_base_stats +
                                     self.turn1_p1_pokemon_name + self.turn1_p1_pokemon_hp_pct + self.turn1_p1_pokemon_status + self.turn1_p1_pokemon_effects + self.turn1_p1_pokemon_boosts + self.turn1_p1_pokemon_move_name + self.turn1_p1_pokemon_move_type + self.turn1_p1_pokemon_move_category + self.turn1_p1_pokemon_move_stats +
                                     self.turn1_p2_pokemon_name + self.turn1_p2_pokemon_hp_pct + self.turn1_p2_pokemon_status + self.turn1_p2_pokemon_effects + self.turn1_p2_pokemon_boosts + self.turn1_p2_pokemon_move_name + self.turn1_p2_pokemon_move_type + self.turn1_p2_pokemon_move_category + self.turn1_p2_pokemon_move_stats +
                                     20822) #the remaining 29 turns
                                     # Add other feature group sizes here

        # Define the input layer based on the specified structure
        self.input_layer = nn.Linear(self.total_input_features, ...) # The rest will be added later
        # Add more layers as needed
        # self.hidden_layer = ...
        # self.output_layer = ...

    def forward(self, x):
        # Define the forward pass
        # x will be a tensor with the input features
        # You can now access feature groups using slicing based on the defined sizes
        # Example: p1_pokemon1_name_features = x[:, :self.p1_pokemon1_name_features]
        # p1_pokemon1_level_feature = x[:, self.p1_pokemon1_name_features : self.p1_pokemon1_name_features + self.p1_pokemon1_level_feature]
        # ... and so on for all features


        x = self.input_layer(x)
        # Add more layers as needed
        return x

# Example of creating an instance of the model (for demonstration)
# model = PokemonBattleModel()
# print(model)

Looking in indexes: https://download.pytorch.org/whl/cu126


Pokemon features from json

In [3]:
import json
import os
import torch

# --- Funzioni di caricamento e codifica (invariate) ---

# Load mappings for one-hot encoding
def load_json_keys_as_list(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
        return list(data.keys())

try:
    # Assicurati che i percorsi siano corretti per il tuo ambiente
    base_path = './fds-challenge-dataset/'
    pokemon_names = load_json_keys_as_list(os.path.join(base_path, 'pokedex.json'))
    move_names = load_json_keys_as_list(os.path.join(base_path, 'moves.json'))
    statuses = load_json_keys_as_list(os.path.join(base_path, 'statuses.json'))
    # Aggiungiamo 'nostatus' se non è presente nel file, dato che è un valore comune
    if 'nostatus' not in statuses:
        statuses.append('nostatus')
    types = [t.lower() for t in load_json_keys_as_list(os.path.join(base_path, 'types.json'))]

    effects = ['noeffect', 'reflect'] # Assicurati che questa lista sia completa
    move_categories = ['PHYSICAL', 'SPECIAL', 'STATUS']

except FileNotFoundError as e:
    print(f"Error loading mapping file: {e}")
    raise

def one_hot_encode(value, categories, case_insensitive=False):
    """Creates a one-hot encoding vector."""
    if case_insensitive:
        value = value.lower()
        categories = [c.lower() for c in categories]

    try:
        index = categories.index(value)
        one_hot = [0.0] * len(categories)
        one_hot[index] = 1.0
        return one_hot
    except ValueError:
        # Fornisce un messaggio di errore più dettagliato
        raise ValueError(f"Value '{value}' not found in the provided categories for one-hot encoding.")

# --- Funzione di supporto per elaborare i dati di un turno per un giocatore ---

def _process_player_turn_state(turn_data, player_prefix):
    """
    Elabora lo stato e la mossa di un singolo giocatore per un turno e restituisce il suo feature vector.
    `player_prefix` dovrebbe essere 'p1' o 'p2'.
    """
    player_features = []

    pokemon_state_key = f'{player_prefix}_pokemon_state'
    move_details_key = f'{player_prefix}_move_details'

    # Calcola la dimensione del vettore per il padding in caso di dati mancanti
    # Nome(151) + HP(1) + Status(len) + Effetti(len) + Boosts(5) + Nome Mossa(165) + Tipo Mossa(20) + Cat Mossa(3) + Stat Mossa(3)
    VECTOR_SIZE_PER_PLAYER_TURN = (len(pokemon_names) + 1 + len(statuses) + len(effects) + 5 +
                                   len(move_names) + len(types) + len(move_categories) + 3)

    if turn_data.get(pokemon_state_key):
        pokemon_state = turn_data[pokemon_state_key]

        # One-hot encoding del nome del pokemon (es. 151 dimensioni)
        player_features.extend(one_hot_encode(pokemon_state['name'], pokemon_names))

        # Percentuale HP (1 dimensione numerica)
        player_features.append(float(pokemon_state['hp_pct']))

        # One-hot encoding dello status (es. 6 dimensioni)
        player_features.extend(one_hot_encode(pokemon_state['status'], statuses))

        # One-hot encoding degli effetti (es. 2 dimensioni)
        effects_one_hot = [0.0] * len(effects)
        for effect in pokemon_state['effects']:
            if effect in effects:
                effects_one_hot[effects.index(effect)] = 1.0
        player_features.extend(effects_one_hot)

        # Boosts (5 dimensioni numeriche: atk, def, spa, spd, spe)
        boost_stats = ['atk', 'def', 'spa', 'spd', 'spe']
        player_features.extend([float(pokemon_state['boosts'].get(stat, 0)) for stat in boost_stats])

        # Dettagli della mossa
        if turn_data.get(move_details_key):
            move_details = turn_data[move_details_key]

            # One-hot encoding del nome della mossa (es. 165 dimensioni)
            player_features.extend(one_hot_encode(move_details['name'], move_names))

            # One-hot encoding del tipo della mossa (es. 20 dimensioni)
            player_features.extend(one_hot_encode(move_details['type'], types, case_insensitive=True))

            # One-hot encoding della categoria della mossa (3 dimensioni)
            player_features.extend(one_hot_encode(move_details['category'], move_categories))

            # Statistiche della mossa (Potenza, Precisione, Priorità) (3 dimensioni)
            player_features.append(float(move_details.get('base_power', 0)))
            player_features.append(float(move_details.get('accuracy', 0)))
            player_features.append(float(move_details.get('priority', 0)))
        else:
            # Aggiungi zeri se non ci sono dettagli sulla mossa
            player_features.extend([0.0] * (len(move_names) + len(types) + len(move_categories) + 3))

        return player_features
    else:
        # Se non ci sono dati per il pokemon del giocatore in questo turno, aggiungi zeri
        return [0.0] * VECTOR_SIZE_PER_PLAYER_TURN


# --- Funzione principale aggiornata ---

def create_feature_vector(data_entry):
    """
    Converts a single JSON data entry into a flattened float feature vector.
    """
    feature_vector = []

    # Process P1's team details (6 Pokemon) - Questa parte rimane invariata
    for pokemon_data in data_entry['p1_team_details']:
        feature_vector.extend(one_hot_encode(pokemon_data['name'], pokemon_names))
        feature_vector.append(float(pokemon_data['level']))
        types_one_hot = [0.0] * (len(types) * 2)
        for i, pokemon_type in enumerate(pokemon_data['types']):
            if pokemon_type.lower() in types:
                type_index = types.index(pokemon_type.lower())
                if i < 2:
                    types_one_hot[type_index + i * len(types)] = 1.0
        feature_vector.extend(types_one_hot)
        feature_vector.append(float(pokemon_data['base_hp']))
        feature_vector.append(float(pokemon_data['base_atk']))
        feature_vector.append(float(pokemon_data['base_def']))
        feature_vector.append(float(pokemon_data['base_spa']))
        feature_vector.append(float(pokemon_data['base_spd']))
        feature_vector.append(float(pokemon_data['base_spe']))

    # Process P2's first Pokemon details - Questa parte rimane invariata
    if data_entry['p2_lead_details']:
        p2_pokemon1_data = data_entry['p2_lead_details']
        feature_vector.extend(one_hot_encode(p2_pokemon1_data['name'], pokemon_names))
        feature_vector.append(float(p2_pokemon1_data['level']))
        p2_types_one_hot = [0.0] * (len(types) * 2)
        for i, pokemon_type in enumerate(p2_pokemon1_data['types']):
            if pokemon_type.lower() in types:
                type_index = types.index(pokemon_type.lower())
                if i < 2:
                    p2_types_one_hot[type_index + i * len(types)] = 1.0
        feature_vector.extend(p2_types_one_hot)
        feature_vector.append(float(p2_pokemon1_data['base_hp']))
        feature_vector.append(float(p2_pokemon1_data['base_atk']))
        feature_vector.append(float(p2_pokemon1_data['base_def']))
        feature_vector.append(float(p2_pokemon1_data['base_spa']))
        feature_vector.append(float(p2_pokemon1_data['base_spd']))
        feature_vector.append(float(p2_pokemon1_data['base_spe']))
    else:
         feature_vector.extend([0.0] * (len(pokemon_names) + 1 + (len(types) * 2) + 6))

    # === SEZIONE MODIFICATA ===
    # Processa i turni dalla battle_timeline
    if 'battle_timeline' in data_entry:
        for turn_data in data_entry['battle_timeline']:
            # Processa lo stato e la mossa di P1 per questo turno
            feature_vector.extend(_process_player_turn_state(turn_data, 'p1'))

            # Processa lo stato e la mossa di P2 per questo turno
            feature_vector.extend(_process_player_turn_state(turn_data, 'p2'))

    return feature_vector

# --- Esempio di utilizzo (invariato) ---

# train_file_path = './fds-challenge-dataset/kfolds/fold_1_train.jsonl'
# feature_vectors = []

# try:
#     with open(train_file_path, 'r') as f:
#         for i, line in enumerate(f):
#             try:
#                 data_entry = json.loads(line)
#                 feature_vector = create_feature_vector(data_entry)
#                 feature_vectors.append(feature_vector)
#             except (KeyError, ValueError) as e:
#                 print(f"Error processing line {i+1} in {train_file_path}: {e}")
#                 # Puoi decidere di saltare la riga o interrompere l'esecuzione
#                 continue

#     if feature_vectors:
#         vector_length = len(feature_vectors[0])
#         if all(len(v) == vector_length for v in feature_vectors):
#              feature_tensor = torch.tensor(feature_vectors, dtype=torch.float32)
#              print(f"Created a tensor with shape: {feature_tensor.shape}")
#         else:
#              print("Error: Feature vectors have inconsistent lengths.")
#              # Debug: stampa la lunghezza dei vettori anomali
#              for i, v in enumerate(feature_vectors):
#                  if len(v) != vector_length:
#                      print(f"Vector {i} has length {len(v)}, expected {vector_length}")
#     else:
#         print("No data entries were successfully processed.")

# except FileNotFoundError:
#     print(f"Error: File not found at {train_file_path}")
# except Exception as e:
#     print(f"An unexpected error occurred: {e}")

Vector encoding test

In [4]:
import json
import random
import torch

# Define the path to the training data file
train_file_path = '/content/drive/MyDrive/fds-challenge-dataset/kfolds/fold_1_train.jsonl'

try:
    # Read all lines from the file
    with open(train_file_path, 'r') as f:
        lines = f.readlines()

    if lines:
        # Select a random line
        random_line = random.choice(lines)

        # Parse the JSON data
        data_entry = json.loads(random_line)

        # Create the feature vector
        feature_vector = create_feature_vector(data_entry)

        # Convert the feature vector to a tensor
        feature_tensor = torch.tensor(feature_vector, dtype=torch.float32)

        # Print the original JSON data (formatted) and the feature vector
        print("--- Random JSON Entry ---")
        print(json.dumps(data_entry, indent=4))
        print("\n--- Feature Vector ---")
        print(feature_tensor)
        print(f"\nFeature Vector Length: {len(feature_vector)}")

    else:
        print("The training file is empty.")

except FileNotFoundError:
    print(f"Error: File not found at {train_file_path}")
except json.JSONDecodeError:
    print(f"Error: Could not decode JSON from the selected line.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

Error: File not found at /content/drive/MyDrive/fds-challenge-dataset/kfolds/fold_1_train.jsonl


MLP definition and utilities

In [5]:
from torch.utils.data import Dataset, DataLoader

class PokemonBattleDataset(Dataset):
    """
    PyTorch Dataset for loading Pokemon battle data from a .jsonl file.
    """
    def __init__(self, file_path):
        self.file_path = file_path
        # Load all lines into memory. For very large files, this could be optimized.
        with open(file_path, 'r') as f:
            self.lines = f.readlines()

    def __len__(self):
        return len(self.lines)

    def __getitem__(self, idx):
        line = self.lines[idx]
        data_entry = json.loads(line)

        # Create feature vector
        features = create_feature_vector(data_entry)

        # Get the label (1.0 for win, 0.0 for loss)
        label = 1.0 if data_entry['player_won'] else 0.0

        # Convert to tensors
        feature_tensor = torch.tensor(features, dtype=torch.float32)
        label_tensor = torch.tensor([label], dtype=torch.float32)

        return feature_tensor, label_tensor

# Example of creating a dataset (we will do this in the training cell)
# train_dataset = PokemonBattleDataset('/content/drive/MyDrive/fds-challenge-dataset/kfolds/fold_1_train.jsonl')
# print(f"Successfully created dataset with {len(train_dataset)} samples.")
# features, label = train_dataset[0]
# print(f"Sample feature shape: {features.shape}")
# print(f"Sample label shape: {label.shape}")

import torch.nn as nn

class BattleMLP(nn.Module):
    def __init__(self, input_size, hidden_size1=512, hidden_size2=256):
        super(BattleMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_size1, hidden_size2),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden_size2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

# Example of creating a model instance
# model = BattleMLP(input_size=INPUT_DIMENSION)
# print(model)

Training loop

In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.notebook import tqdm  # Use tqdm.notebook for better Jupyter integration
import json
import os

# ### NEW ### Import libraries for interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- Hyperparameters and Setup ---
LEARNING_RATE = 0.0001
BATCH_SIZE = 64
EPOCHS = 10
FOLD_TO_TRAIN = 1
INPUT_DIMENSION = 22866
CHECKPOINT_DIR = './pokemon_battle_checkpoints/'

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- (Your PokemonBattleDataset and BattleMLP class definitions would go here) ---
# Placeholder classes for demonstration purposes
class PokemonBattleDataset(torch.utils.data.Dataset):
    def __init__(self, file_path):
        self.length = 10000 if 'train' in file_path else 1000
    def __len__(self):
        return self.length
    def __getitem__(self, idx):
        return torch.randn(INPUT_DIMENSION), torch.randint(0, 2, (1,)).float().squeeze()

class BattleMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.linear = nn.Linear(input_size, 1)
    def forward(self, x):
        return self.linear(x)
# ---------------------------------------------------------------------------------

# --- Prepare DataLoaders ---
train_file = f'./fds-challenge-dataset/kfolds/fold_{FOLD_TO_TRAIN}_train.jsonl'
val_file = f'./fds-challenge-dataset/kfolds/fold_{FOLD_TO_TRAIN}_val.jsonl'
train_dataset = PokemonBattleDataset(train_file)
val_dataset = PokemonBattleDataset(val_file)
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# --- Initialize Model, Loss, and Optimizer ---
model = BattleMLP(input_size=INPUT_DIMENSION).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ### NEW ### Wrap the entire training process in a function
def execute_training_loop(start_epoch):
    """Contains the main training and validation logic."""
    for epoch in range(start_epoch, EPOCHS):
        # --- Training Phase ---
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for i, (features, labels) in enumerate(train_pbar):
            features, labels = features.to(device), labels.to(device)
            outputs = model(features)
            loss = loss_fn(outputs.squeeze(), labels.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            predicted = (outputs.squeeze() > 0).float()
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            running_loss = train_loss / (i + 1)
            running_acc = 100 * train_correct / train_total
            train_pbar.set_postfix({'loss': f'{running_loss:.4f}', 'acc': f'{running_acc:.2f}%'})

        # --- Validation Phase ---
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]")
        with torch.no_grad():
            for i, (features, labels) in enumerate(val_pbar):
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                loss = loss_fn(outputs.squeeze(), labels.float())
                val_loss += loss.item()
                predicted = (outputs.squeeze() > 0).float()
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                running_loss = val_loss / (i + 1)
                running_acc = 100 * val_correct / val_total
                val_pbar.set_postfix({'loss': f'{running_loss:.4f}', 'acc': f'{running_acc:.2f}%'})

        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{EPOCHS} Summary | Val Loss: {avg_val_loss:.4f}, Val Acc: {100 * val_correct / val_total:.2f}%")

        # --- Save Checkpoint ---
        save_path = os.path.join(CHECKPOINT_DIR, f'model_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_val_loss,
        }, save_path)
        print(f"Checkpoint saved to {save_path}\n")
    print("Training finished.")

def run_training_from_selection(b):
    """Callback function to load checkpoint and start training."""
    # Clear the dropdown and button from the output
    clear_output()
    
    choice = dropdown.value
    start_epoch = 0
    
    if choice == 'Start new training from scratch':
        print("No checkpoint selected. Starting new training from scratch.")
    else:
        checkpoint_path = os.path.join(CHECKPOINT_DIR, choice)
        print(f"Loading checkpoint: {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # ### FIX ###: Add logic to handle both old and new checkpoint formats
        # Check if the loaded object is a dictionary and has our expected key.
        if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
            # This is the new, dictionary-based format
            print("Loading from dictionary-based checkpoint format.")
            model.load_state_dict(checkpoint['model_state_dict'])
            # Also load optimizer and epoch if they exist
            if 'optimizer_state_dict' in checkpoint:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if 'epoch' in checkpoint:
                start_epoch = checkpoint['epoch']
            print(f"Successfully loaded. Resuming training from Epoch {start_epoch + 1}.")
        else:
            # This is the old format, where the file is just the state_dict
            print("Loading from legacy state_dict-only checkpoint format.")
            print("Warning: Optimizer state and epoch number not found. Optimizer will be reset.")
            model.load_state_dict(checkpoint)
            start_epoch = 0 # Can't resume epoch, so we start fresh
            print(f"Successfully loaded model weights. Starting training from Epoch 1.")
            
    # Finally, call the main training loop
    execute_training_loop(start_epoch)

# ### NEW ### CHECKPOINT SELECTION UI
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoints = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pth')])
options = ['Start new training from scratch'] + checkpoints

# Create the interactive widgets
dropdown = widgets.Dropdown(
    options=options,
    description='Select Checkpoint:',
    style={'description_width': 'initial'},
    layout={'width': 'max-content'}
)

button = widgets.Button(
    description='Load and Continue Training',
    button_style='success',
    tooltip='Click to start the training process'
)

# Register the function to run on button click
button.on_click(run_training_from_selection)

# Display the widgets. The code will wait here until the button is clicked.
print("Please make a selection and click the button to proceed.")
display(dropdown, button)

RuntimeError: Error(s) in loading state_dict for BattleMLP:
	Missing key(s) in state_dict: "linear.weight", "linear.bias". 
	Unexpected key(s) in state_dict: "network.0.weight", "network.0.bias", "network.3.weight", "network.3.bias", "network.6.weight", "network.6.bias". 

Train loop on all data

In [ ]:
import torch.optim as optim
from tqdm import tqdm

# --- Hyperparameters and Setup ---
LEARNING_RATE = 0.0001
BATCH_SIZE = 64
EPOCHS = 15  # Increased to 15 epochs
INPUT_DIMENSION = 22866 # Ensure this matches your feature vector size

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Prepare DataLoaders ---
# Use the full training data
train_file = './train.jsonl'
train_dataset = PokemonBattleDataset(train_file)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- Initialize Model, Loss, and Optimizer ---
# Initialize a new model or ensure the existing one is reset if needed
model = BattleMLP(input_size=INPUT_DIMENSION).to(device)
loss_fn = nn.BCELoss() # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# --- Training Loop ---
for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
    for features, labels in train_pbar:
        features, labels = features.to(device), labels.to(device)

        # Forward pass
        outputs = model(features)
        loss = loss_fn(outputs, labels)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Calculate metrics
        train_loss += loss.item()
        predicted = (outputs > 0.5).float()
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        # Update progress bar
        train_pbar.set_postfix({'loss': f'{train_loss/(train_pbar.n+1):.4f}', 'acc': f'{100*train_correct/train_total:.2f}%'})


    avg_train_loss = train_loss / len(train_loader)
    train_accuracy = 100 * train_correct / train_total

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%")

print("Training finished.")

In [5]:
# ==============================================================================
# CELL 1: ONE-TIME DATA PRE-PROCESSING
# This cell converts the raw .jsonl file into optimized NumPy arrays for fast training.
# WARNING: This is I/O and CPU intensive and will take a long time to run.
# It will also create very large files on disk (~1.2 TB).
# ==============================================================================
import numpy as np
import json
from tqdm.notebook import tqdm
import os

def preprocess_data():
    # --- Configuration ---
    input_jsonl_path = './fds-challenge-dataset/train.jsonl'
    output_dir = './fds-challenge-dataset/preprocessed'
    output_features_path = os.path.join(output_dir, 'train_features.npy')
    output_labels_path = os.path.join(output_dir, 'train_labels.npy')
    
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # --- Get total number of samples and feature dimension ---
    print("Step 1: Counting lines in the input file...")
    with open(input_jsonl_path, 'r') as f:
        num_samples = sum(1 for _ in f)
    
    # This must match the output of your create_feature_vector function
    feature_dimension = 22866 
    print(f"Found {num_samples:,} samples.")
    print(f"Feature dimension is {feature_dimension}.")

    # --- Create memory-mapped NumPy arrays on disk ---
    # These arrays reside on disk and are treated like they are in memory.
    print(f"Step 2: Creating memory-mapped files at '{output_dir}'...")
    # 'w+' mode creates the file or overwrites it if it exists.
    features_memmap = np.memmap(output_features_path, dtype=np.float32, mode='w+', shape=(num_samples, feature_dimension))
    labels_memmap = np.memmap(output_labels_path, dtype=np.float32, mode='w+', shape=(num_samples, 1))
    print("Memory-mapped files created successfully.")

    # --- Loop through the data and fill the arrays ---
    print("Step 3: Processing JSONL file and populating arrays...")
    with open(input_jsonl_path, 'r') as f:
        for i, line in enumerate(tqdm(f, total=num_samples, desc="Processing samples")):
            try:
                data_entry = json.loads(line)
                
                # Generate features and label using your existing function
                feature_vector = create_feature_vector(data_entry)
                label = 1.0 if data_entry['player_won'] else 0.0
                
                # Write to the memory-mapped arrays
                features_memmap[i] = feature_vector
                labels_memmap[i] = label
                
            except Exception as e:
                print(f"Error processing line {i+1}: {e}. Filling with zeros.")
                features_memmap[i] = np.zeros(feature_dimension, dtype=np.float32)
                labels_memmap[i] = 0.0
                continue

    # --- Finalize ---
    print("Step 4: Flushing data to disk...")
    features_memmap.flush()
    labels_memmap.flush()
    print("Pre-processing complete! You can now run the training cell.")

# Execute the pre-processing
preprocess_data()

Step 1: Counting lines in the input file...
Found 7,200,000 samples.
Feature dimension is 22866.
Step 2: Creating memory-mapped files at './fds-challenge-dataset/preprocessed'...


OSError: [Errno 28] No space left on device

<h2>Train on all AUGMENTED data</h2>

In [ ]:
# ==============================================================================
# CELL 2: HIGH-PERFORMANCE TRAINING LOOP
# This cell trains the model using the pre-processed data for maximum speed.
# ==============================================================================
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import os
import numpy as np
import gc

# --- 1. New, Ultra-Fast Dataset for Pre-processed Data ---
class PreprocessedDataset(Dataset):
    """
    A very fast PyTorch Dataset that reads from pre-processed NumPy memory-mapped files.
    `__getitem__` is just a quick array slice, making it ideal for multiprocessing.
    """
    def __init__(self, features_path, labels_path):
        # 'r' mode opens the files for reading without loading them into RAM
        self.features = np.load(features_path, mmap_mode='r')
        self.labels = np.load(labels_path, mmap_mode='r')
        print(f"Initialized dataset with {len(self.features):,} pre-processed samples.")

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        # This is extremely fast. The OS handles loading the data from disk.
        feature = self.features[idx]
        label = self.labels[idx]
        # Note: Conversion to torch.tensor happens automatically in the DataLoader
        return feature, label

# --- 2. Main Training Function ---
def run_training_from_preprocessed():
    # --- Hyperparameters and Setup ---
    LEARNING_RATE = 0.0001
    BATCH_SIZE = 256
    EPOCHS = 15
    INPUT_DIMENSION = 22866
    NUM_WORKERS = 4 # Keep using workers, they will be very fast now

    # Setup device
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")

    # --- Prepare DataLoaders using the NEW Dataset ---
    features_file = './fds-challenge-dataset/preprocessed/train_features.npy'
    labels_file = './fds-challenge-dataset/preprocessed/train_labels.npy'
    
    train_dataset = PreprocessedDataset(features_file, labels_file)

    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False,
        persistent_workers=True if NUM_WORKERS > 0 else False
    )

    # --- Initialize Model, Loss, and Optimizer ---
    model = BattleMLP(input_size=INPUT_DIMENSION).to(device)
    loss_fn = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    # --- Training Loop ---
    print("\nStarting training...")
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
        for features, labels in train_pbar:
            features, labels = features.to(device, non_blocking=True), labels.to(device, non_blocking=True)

            outputs = model(features)
            loss = loss_fn(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            predicted = (outputs > 0.5).float()
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

            train_pbar.set_postfix({'loss': f'{train_loss/(train_pbar.n+1):.4f}', 'acc': f'{100*train_correct/train_total:.2f}%'})

        avg_train_loss = train_loss / len(train_loader)
        train_accuracy = 100 * train_correct / train_total

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%")

    print("Training finished.")

# --- 3. Execute the Training ---
run_training_from_preprocessed()

Using device: cuda
Loading existing index from: ./fds-challenge-dataset/train.jsonl.index.pkl
Dataset initialized with 7,200,000 samples.


Epoch 1/15 [Train]:   0%|          | 0/28125 [00:00<?, ?it/s]

Save model to drive

In [9]:
import torch
import os

# Define the directory to save checkpoints
checkpoint_dir = './pokemon_battle_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Define the path for the checkpoint file
checkpoint_path = os.path.join(checkpoint_dir, f'battle_mlp_fold_{FOLD_TO_TRAIN}_epoch_{epoch+1}.pth')

# Save the model's state dictionary
torch.save(model.state_dict(), checkpoint_path)

print(f"Model checkpoint saved to: {checkpoint_path}")

Model checkpoint saved to: ./pokemon_battle_checkpoints\battle_mlp_fold_1_epoch_3.pth


Test inference

In [ ]:
import json
import pandas as pd
import torch

# Define the path to the test data file and the output submission file
test_file_path = '/content/drive/MyDrive/fds-challenge-dataset/test.jsonl'
submission_file_path = '/content/drive/MyDrive/pokemon_battle_submission.csv'

# Load the trained model
# Assuming the model object 'model' is still available from the training cell
# If not, you would need to load it from the saved checkpoint:
# model = BattleMLP(input_size=INPUT_DIMENSION).to(device)
# model.load_state_dict(torch.load(checkpoint_path))
# model.eval() # Set the model to evaluation mode

# Ensure the model is on the correct device and in evaluation mode
model.to(device)
model.eval()

# Lists to store battle IDs and predictions
battle_ids = []
predictions = []

print(f"Processing test data from {test_file_path}...")

try:
    with open(test_file_path, 'r') as f:
        for i, line in enumerate(f):
            try:
                data_entry = json.loads(line)

                # Extract battle_id
                battle_id = data_entry.get('battle_id')
                if battle_id is None:
                    print(f"Warning: 'battle_id' not found in line {i+1}. Skipping.")
                    continue

                # Create feature vector (using the previously defined function)
                # Note: The test data does not have 'player_won', so create_feature_vector should handle this.
                # Make sure your create_feature_vector function doesn't rely on 'player_won'.
                feature_vector = create_feature_vector(data_entry)
                feature_tensor = torch.tensor([feature_vector], dtype=torch.float32).to(device)

                # Perform inference
                with torch.no_grad():
                    output = model(feature_tensor)
                    # Get the predicted class (0 or 1)
                    predicted_class = (output > 0.5).int().item()

                # Store battle_id and prediction
                battle_ids.append(battle_id)
                predictions.append(predicted_class)

            except (KeyError, ValueError, json.JSONDecodeError) as e:
                print(f"Error processing line {i+1} in {test_file_path}: {e}")
                # You can decide to skip the line or handle the error differently
                continue
            except Exception as e:
                print(f"An unexpected error occurred while processing line {i+1}: {e}")
                continue


    # Create a pandas DataFrame for the submission file
    submission_df = pd.DataFrame({'battle_id': battle_ids, 'player_won': predictions})

    # Save the DataFrame to a CSV file
    submission_df.to_csv(submission_file_path, index=False)

    print(f"Submission file created successfully at {submission_file_path}")
    print(submission_df.head()) # Display the first few rows of the submission file

except FileNotFoundError:
    print(f"Error: Test file not found at {test_file_path}")
except Exception as e:
    print(f"An unexpected error occurred during inference: {e}")

Processing test data from /content/drive/MyDrive/fds-challenge-dataset/test.jsonl...
Submission file created successfully at /content/drive/MyDrive/pokemon_battle_submission.csv
   battle_id  player_won
0          0           0
1          1           1
2          2           1
3          3           1
4          4           1
